In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("..")
SILVER_DIR = BASE / "silver_data"
REPORT_DIR = BASE / "reports"
SILVER_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

def profiling(df, table_name):
    print(f"=== PROFILING {table_name} ===")
    print("Shape:", df.shape)
    print("\nMissing:\n", df.isna().sum())
    print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
df = pd.read_csv("../geography.csv")
profiling(df, "GEOGRAPHY")

=== PROFILING GEOGRAPHY ===
Shape: (39948, 4)

Missing:
 zip         0
city        0
region      0
district    0
dtype: int64

Duplicate rows: 0


## 1. Missing Data

In [ ]:
missing_pk = df[df["zip"].isna()].copy()
print("Missing zip:", len(missing_pk))
if len(missing_pk):
    missing_pk.to_csv(REPORT_DIR / "_rejected_geography_missing_zip.csv", index=False)
    df = df[df["zip"].notna()].copy()

print("Remaining missing by column:\n", df.isna().sum())

Missing zip: 0
Remaining missing by column:
 zip         0
city        0
region      0
district    0
dtype: int64


## 2. Outlier / Domain rule

In [ ]:
print("zip âm:", (pd.to_numeric(df["zip"], errors="coerce") < 0).sum())
print("zip trùng:", df["zip"].duplicated().sum())

zip âm: 0
zip trùng: 0


## 3. Chuẩn hóa kiểu dữ liệu và text

In [ ]:
for col in ["city", "region", "district"]:
    df[col] = df[col].astype("string").str.strip()

df["zip"] = pd.to_numeric(df["zip"], errors="coerce").astype("Int64")

print(df.dtypes)

zip          Int64
city        string
region      string
district    string
dtype: object


## 4. Validation

In [ ]:
assert df["zip"].notna().all(), "GEOGRAPHY: zip còn thiếu"
assert df["zip"].is_unique, "GEOGRAPHY: zip bị trùng"
for col in ["city", "region", "district"]:
    assert df[col].notna().all(), f"GEOGRAPHY: {col} còn thiếu"

print("GEOGRAPHY: validation đạt yêu cầu Silver.")

GEOGRAPHY: validation đạt yêu cầu Silver.


In [ ]:
df.to_csv(SILVER_DIR / "GEOGRAPHY.csv", index=False)
log = [
    ["GEOGRAPHY", "Profiling", len(df), "Đã kiểm tra shape, missing, duplicate"],
    ["GEOGRAPHY", "Missing", len(df), "Không tự sinh zip; cô lập nếu có khóa thiếu"],
    ["GEOGRAPHY", "Inconsistency/type", len(df), "Chuẩn hóa text; zip giữ kiểu Int64 (không ép string)"],
    ["GEOGRAPHY", "Validation", len(df), "PK zip không thiếu/không trùng; cột mô tả không thiếu"],
]
pd.DataFrame(log, columns=["table","step","rows_after","result"]).to_csv(REPORT_DIR / "geography_log.csv", index=False)
print("Đã xuất:", SILVER_DIR / "GEOGRAPHY.csv")

Đã xuất: ../silver_data/GEOGRAPHY.csv
